# Optimising rating factors over several years with `open-rating-engine`

Notebook 02 tunes the `risk.demand_multiplier` rating tables for **one year of new business**. Most of a motor book is renewals, and a renewal price has consequences beyond the year it's charged:

- **This year's premium is next year's previous premium.** The renewal demand model reacts to the year-on-year price change, and the cap and floor limit how far a renewal premium can move from it. A price rise today makes next year's renewal both less likely to convert and harder to reprice.
- **Customers move between rating bands.** Drivers get older and tenure grows, so a vehicle priced by the 0-5 tenure table this year may sit in the 6-10 table in a few years.
- **Lapsed customers don't come back.** A vehicle that doesn't renew earns nothing in any later year.

`PriceOptimizerRatingFactorMultiPeriod` accounts for all three. It searches over the same rating-table multipliers as notebook 02, but scores each candidate by simulating the book forward year by year and discounting the expected profit back to today. It maximises that net present value (NPV) while holding the book's conversion rate.

| Step | What it does |
|---|---|
| Input data | Loads the origination book (new-business quotes) and the in-force book, and ages the in-force book to its next renewal |
| Aging rules | Defines how a policy changes in one year; the optimiser applies them once per simulated year |
| Baseline | Builds the optimiser from the pipeline and simulates five years at today's rates |
| Optimise | Maximises five-year NPV with lifetime conversion held at or above the baseline |
| Efficient frontier | Repeats the optimisation across a range of conversion targets |
| Risk | Samples renew/lapse outcomes to give the distribution of NPV, not just its expected value |
| Deploy | Writes the optimised multipliers back into the pipeline as a `RatingTablesLayer`, checks it against the simulation and saves the pipeline |

**How the simulation works**

Each simulated year is one score of the pipeline on that year's book:

- **Year 0** is next year's quotes: the origination book plus the in-force book at its next renewal.
- **Each later year** ages the whole of the previous year's book by one year. Last year's new business renews alongside the existing renewals. Every carried vehicle takes last year's simulated premium as its previous premium, and a survival weight equal to the chance it has renewed every year so far.
- **New business** in this notebook is a *going concern*: a fresh copy of the origination book is quoted every year. (`new_business_mode="runoff"` would value today's book and its renewals only.)

The NPV is the sum over years and vehicles of `weight × expected profit / (1 + r)^(t + 0.5)`, with profit taken as arriving mid-year.

Unlike notebook 02 there is no precomputed price grid to interpolate. The optimiser runs the real pipeline over the full horizon for every candidate set of multipliers, so each evaluation costs several full scores of the book.

**Prerequisites**

- Run `01_rating_algorithm_example.ipynb` first. It saves the baseline pipeline to `models/rating_engines/pricing_pipeline.pkl`. Notebooks 02 and 03 aren't needed: this notebook starts from the same baseline as 02 and saves its result under its own name.
- Install the package with the ONNX and notebook extras. From the `open-rating-engine` folder: `pip install -e ".[onnx,notebook]"`
- The demand model is an H2O GBM converted to ONNX (`models/motor_demand_onnx/h2o_gbm.onnx`), so no Java runtime is needed.
- Run the notebook from the `.examples` folder, so the `data/` and `models/` folders resolve.
- Sections 6 and 7 are slow: every evaluation is a full five-year simulation, and each optimisation needs a few hundred. Section 5 prints the time one simulation takes on your machine.

In [ ]:
import time
from copy import deepcopy
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlflow.pyfunc import PythonModelContext

from open_rating_engine.data_processing import apply_aging
from open_rating_engine.optimization import PriceOptimizerRatingFactorMultiPeriod
from open_rating_engine.rating_engine import SourceColumn

## 1. Input data

Two books:

- **Origination**: new-business quotes (`policy.transaction_type == "NB"`), the same book notebook 02 optimises. It's quoted again each simulated year.
- **In force**: this year's policies, both renewals and those written as new business. `risk.gwp` is the premium each vehicle is paying this year.

### Sampling the books

Every evaluation of the objective scores the pipeline once per simulated year, on a book that grows each year as new business is added. On the full books that's around half a minute per evaluation, and the optimiser needs a few hundred. The notebook therefore works on a random sample of policies. A policy's vehicles, covers and claims stay together, so a sampled policy is complete. Set `SAMPLE_FRACTION = 1.0` to run on the full books.

In [ ]:
DATA_DIR = Path("data").resolve()
TABLES = ["policy", "risk", "cover", "claims_history"]
SAMPLE_FRACTION = 0.01


def load_book(folder):
    return {table: pd.read_parquet(DATA_DIR / folder / f"{table}.parquet") for table in TABLES}


def sample_policies(book, fraction, seed=42):
    """Keep a random `fraction` of policies, with all of their vehicles, covers and claims."""
    if fraction >= 1.0:
        return book
    keep = book["policy"]["policy.policy_number"].sample(frac=fraction, random_state=seed)
    return {
        table: df[df[f"{table}.policy_number"].isin(keep)].reset_index(drop=True)
        for table, df in book.items()
    }


origination = sample_policies(load_book("origination"), SAMPLE_FRACTION)
in_force = sample_policies(load_book("renewal"), SAMPLE_FRACTION)

pd.DataFrame({
    "origination": {table: len(df) for table, df in origination.items()},
    "in force": {table: len(df) for table, df in in_force.items()},
})

In [ ]:
pd.DataFrame({
    "origination": origination["policy"]["policy.transaction_type"].value_counts(),
    "in force": in_force["policy"]["policy.transaction_type"].value_counts(),
}).fillna(0).astype(int)

## 2. Aging rules

The aging rules say how each source column changes over **one** year. The optimiser applies them once per simulated year, so a vehicle five years out has been aged five times. They're the same rules as notebook 03:

| Column | Next year |
|---|---|
| `risk.vehicle_age`, `risk.driver_youngest_age`, `policy.customer_tenure` | + 1 |
| `policy.effective_date`, `policy.expiry_date` | + 1 year |
| `policy.transaction_type` | `"RN"` for every policy |
| `risk.previous_gwp` | this year's premium, `risk.gwp` |
| `risk.vehicle_market_value` | × 0.95 |

`apply_aging` takes the rules as `SourceColumn` layer configs, each with an `aging_rule` holding a pandas expression. It evaluates each expression with `df` set to the column's table and writes the result back to that column.

The `risk.previous_gwp` rule is the price-change feedback. Inside the simulation the optimiser doesn't evaluate it from the input data. It sets each carried vehicle's previous premium to the premium the simulation charged it the year before, so the feedback follows whatever multipliers are being tried.

New business has its own rules: only the policy dates roll forward, so each year's fresh quotes land in the right calendar year. Their risk attributes stay at the origination book's values.

In [ ]:
def aging_rule(table, column, expression, dtype):
    config = SourceColumn(name=column, source_table=table, source_column=column, dtype=dtype).to_dict()
    config["aging_rule"] = {"expression": expression, "enabled": True}
    return config


DATE_RULES = [
    aging_rule(
        "policy", "policy.effective_date",
        "pd.to_datetime(df['policy.effective_date']) + pd.DateOffset(years=1)", "date",
    ),
    aging_rule(
        "policy", "policy.expiry_date",
        "pd.to_datetime(df['policy.expiry_date']) + pd.DateOffset(years=1)", "date",
    ),
]

RENEWAL_AGING_RULES = {
    rule["name"]: rule
    for rule in [
        aging_rule("risk", "risk.vehicle_age", "df['risk.vehicle_age'] + 1", "numeric"),
        aging_rule("risk", "risk.driver_youngest_age", "df['risk.driver_youngest_age'] + 1", "numeric"),
        aging_rule("policy", "policy.customer_tenure", "df['policy.customer_tenure'] + 1", "numeric"),
        *DATE_RULES,
        aging_rule("policy", "policy.transaction_type", '"RN"', "categorical"),
        aging_rule("risk", "risk.previous_gwp", "df['risk.gwp']", "numeric"),
        aging_rule("risk", "risk.vehicle_market_value", "df['risk.vehicle_market_value'] * 0.95", "numeric"),
    ]
}

NEW_BUSINESS_AGING_RULES = {rule["name"]: rule for rule in DATE_RULES}

### The two cohorts in year 0

The optimiser takes the two cohorts separately:

- `data_nb`, the new-business quotes: the origination book as it is.
- `data_rn`, the renewal quotes: the in-force book aged once to its next renewal, exactly as in notebook 03. `apply_aging` ages the frames in place, so it gets a copy.

In [ ]:
data_nb = origination
data_rn = apply_aging(RENEWAL_AGING_RULES, {table: df.copy() for table, df in in_force.items()})

pd.DataFrame({
    column: {
        "In force": in_force[rule["source_table"]][column].iloc[0],
        "Year 0 renewal": data_rn[rule["source_table"]][column].iloc[0],
    }
    for column, rule in RENEWAL_AGING_RULES.items()
}).T

## 3. Baseline rating engine

The baseline pipeline from notebook 01, with every demand multiplier at 1.0. As in notebook 02, a fresh context built from this folder replaces the saved one, whose artifact paths point at the machine that saved the pipeline.

In [ ]:
MODELS_DIR = Path("models").resolve()
BASELINE_PATH = MODELS_DIR / "rating_engines" / "pricing_pipeline.pkl"

with BASELINE_PATH.open("rb") as file:
    pricing_engine = cloudpickle.load(file)

artifacts = {
    "motor_risk_cost_onnx": MODELS_DIR / "motor_risk_cost_onnx" / "pipeline_onnx.onnx",
    "motor_demand_onnx": MODELS_DIR / "motor_demand_onnx" / "h2o_gbm.onnx",
}
missing = [key for key, path in artifacts.items() if not path.exists()]
assert not missing, f"Model files not found for {missing}. Run this notebook from open-rating-engine/.examples."

context = PythonModelContext(
    artifacts={key: str(path) for key, path in artifacts.items()},
    model_config={"environment": "local"},
)
pricing_engine.load_context(context)

# The optimiser scores the book hundreds of times; one progress bar per score is noise.
pricing_engine.options["show_progress"] = False

print(f"{len(pricing_engine.get_layer_names())} layers loaded")

### The rating factors to optimise

The same decision variables as notebook 02: `risk.demand_multiplier` multiplies a youngest-driver age-band table by a customer tenure-band table, giving 9 rows. One set of tables prices both new business and renewals, in every simulated year.

In [ ]:
LAYER_NAME = "risk.demand_multiplier"

demand_layer = pricing_engine.layers[LAYER_NAME]
print(f"formula: {demand_layer.formula}")
print(f"default_values: {demand_layer.default_values}")

for name, table in demand_layer.rating_tables.items():
    print(f"\n{name}")
    display(table)

### Score year 0

`from_pipeline` reads each vehicle's rating-table keys from the scored books, so both cohorts are scored once up front. The KPIs show why the renewal book matters: renewals convert far more often than new-business quotes, and make up most of the expected profit.

In [ ]:
scored_nb = pricing_engine.predict(data_nb)
scored_rn = pricing_engine.predict(data_rn)

KPI_COLUMNS = {
    "Offered GWP": "risk.gwp",
    "Conversion rate": "risk.conversion_rate",
    "Expected GWP": "risk.reporting_expected_gwp",
    "Expected profit": "risk.reporting_expected_profit",
}


def book_kpis(scored):
    risk = scored["risk"]
    kpis = {label: risk[column].mean() for label, column in KPI_COLUMNS.items()}
    kpis["Expected profit margin"] = kpis["Expected profit"] / kpis["Expected GWP"]
    kpis["Vehicles"] = len(risk)
    kpis["Total expected profit"] = risk["risk.reporting_expected_profit"].sum()
    return pd.Series(kpis)


pd.DataFrame({
    "New business": book_kpis(scored_nb),
    "Renewals": book_kpis(scored_rn),
}).round(4)

## 4. Build the optimiser

`from_pipeline` reads the decision space from the layer, as in notebook 02: each rating table becomes a rating factor and its rows become the categories. It keeps the pipeline, which it deep-copies and scores for every evaluation.

The horizon settings:

- `n_periods=5` simulates five years.
- `discount_rate=0.10` discounts each year's profit at 10%. `timing_offset` defaults to 0.5, so profit is taken as arriving mid-year.
- `new_business_mode="going_concern"` quotes a fresh copy of `data_nb` every year. `nb_growth_rate` would grow that cohort by a fixed rate each year; it's left at 0.
- `id_columns` makes each year's new-business vehicles distinct from the carried copies of the same vehicles. Without it, year 2's fresh quote of a policy would share its `policy_number` with the year-1 policy that's now renewing, and the pipeline's policy-to-risk joins would mix the two. Every column a key appears in is listed, foreign keys included, so the joins still work within each year's cohort.

The `*_col` arguments name the scored columns the optimiser reads. The survival weight is written to a `sample_weights` column on the risk table.

### The objective

`objective_type` accepts a named metric, or any function of the simulation's results. Here the objective is the NPV divided by today's total expected profit, which is the horizon NPV measured in *years of today's profit*. It has the same optimum as the raw NPV, and the summary still reports the NPV itself in its `npv` column.

The rescaling matters for the solver settings in section 6. The NPV is in the millions and grows with `SAMPLE_FRACTION`, while the conversion constraint is below 1. Measured in years of profit, the objective is of order 1 at any sample size, so the constraint penalty weight and the convergence tolerance can be fixed numbers.

In [ ]:
N_PERIODS = 5
DISCOUNT_RATE = 0.10

annual_expected_profit = (
    scored_nb["risk"]["risk.reporting_expected_profit"].sum()
    + scored_rn["risk"]["risk.reporting_expected_profit"].sum()
)


def npv_in_years_of_profit(agg):
    """Horizon NPV of expected profit, as a multiple of today's total expected profit."""
    return agg.npv / annual_expected_profit


optimizer = PriceOptimizerRatingFactorMultiPeriod.from_pipeline(
    pipeline=pricing_engine,
    layer_name=LAYER_NAME,
    scored_data_nb=scored_nb,
    scored_data_rn=scored_rn,
    data_nb=data_nb,
    data_rn=data_rn,
    n_periods=N_PERIODS,
    aging_rules_rn=RENEWAL_AGING_RULES,
    aging_rules_nb=NEW_BUSINESS_AGING_RULES,
    new_business_mode="going_concern",
    discount_rate=DISCOUNT_RATE,
    objective_type=npv_in_years_of_profit,
    id_columns={
        "policy_number": [
            ("policy", "policy.policy_number"),
            ("risk", "risk.policy_number"),
            ("cover", "cover.policy_number"),
            ("claims_history", "claims_history.policy_number"),
        ],
        "risk_number": [
            ("risk", "risk.risk_number"),
            ("cover", "cover.risk_number"),
        ],
    },
    conversion_col="risk.conversion_rate",
    profit_col="risk.reporting_expected_profit",
)

print(f"decision variables: {optimizer.input.param_total}")
print(f"factors: {optimizer.input.factor_names}")

## 5. The baseline over five years

`baseline()` simulates the horizon at today's rates without optimising. Its lifetime conversion is the quantity `optimize` constrains: the conversion rate across every quote in every year, each weighted by its survival weight. It sits above the year-0 rate because the book shifts towards renewals over time. New business is the same size every year, while the renewal stock grows by the new business that converts.

The first call also ages the book through all five years. That aging doesn't depend on the multipliers, so it's cached and every later evaluation reuses it.

In [ ]:
start = time.perf_counter()
baseline = optimizer.baseline()
seconds_per_simulation = time.perf_counter() - start

baseline_conversion = baseline["conversion"]
baseline_npv = baseline["npv"]

print(f"one five-year simulation: {seconds_per_simulation:.1f}s")
print(f"lifetime conversion:      {baseline_conversion:.2%}")
print(f"five-year NPV:            ${baseline_npv:,.0f} ({baseline_npv / annual_expected_profit:.2f} years of today's profit)")

The per-year breakdown. `weight` is the expected number of vehicles quoted that year, split into the fresh new business and the renewal stock carried in. `volume` is the expected number that convert, and `npv` is the year's discounted expected profit.

In [ ]:
baseline["per_period_summary"].round(4)

### One vehicle through the horizon

The simulated book has one row per vehicle per year. Following a single renewal shows the mechanics: tenure grows into the next band, last year's premium becomes this year's previous premium, and the survival weight shrinks by each year's conversion rate.

In [ ]:
history = baseline["output_dataframe"]

year0_renewals = history[(history["__period"] == 0) & (history["__cohort"] == "rn")]
example_vehicle = year0_renewals.loc[year0_renewals["risk.customer_tenure"] == 5, "risk.risk_number"].iloc[0]

history.loc[
    history["risk.risk_number"] == example_vehicle,
    [
        "__period", "risk.customer_tenure", "risk.customer_tenure_band", "risk.risk_age_band",
        "sample_weights", "risk.previous_gwp", "risk.gwp", "risk.conversion_rate",
        "risk.reporting_expected_profit", "expected_npv",
    ],
].set_index("__period").round(4)

### Uniform price changes

Scaling every multiplier by the same factor shows how the five-year NPV and lifetime conversion respond to price, and gives a line to compare the optimised result against in section 7. The age-band table is scaled and the tenure table left at 1.0, so every vehicle's multiplier is the factor. Each point is one full five-year simulation.

In [ ]:
UNIFORM_FACTORS = [0.90, 0.95, 1.00, 1.05, 1.10]

age_offset = optimizer.input.param_offsets["risk_age_band"]
age_rows = len(optimizer.input.category_orders["risk_age_band"])


def uniform_overlay(factor):
    # Decision vector with every age-band value scaled by `factor`, tenure values unchanged.
    x = optimizer.input.initial_flat.copy()
    x[age_offset:age_offset + age_rows] *= factor
    return x


uniform_rows = []
for factor in UNIFORM_FACTORS:
    run = baseline if factor == 1.0 else optimizer.baseline(uniform_overlay(factor))
    uniform_rows.append({"factor": factor, "conversion": run["conversion"], "npv": run["npv"]})

uniform_response = pd.DataFrame(uniform_rows).set_index("factor")
uniform_response.round(4)

## 6. Optimise

### Maximise five-year NPV at today's conversion

`optimize` maximises the objective subject to a lifetime constraint. `constraint_kind="floor"` makes it a lower bound, so the optimiser can raise conversion where that pays but can't trade it away. (`"target"` would hold it exactly, and `constraint_metric` can constrain another quantity, such as `"total_volume"`.)

**Nelder-Mead.** Nelder-Mead searches using objective values alone, with no gradients. That suits this problem: the demand model is a GBM, which is piecewise constant in price, so finite-difference gradients are noisy unless their step is tuned. It keeps a simplex of 10 candidate tables (one more than the 9 decision variables) and replaces the worst candidate at each step. Most steps cost one or two simulations.

Nelder-Mead can't enforce a constraint directly. `optimize` adds `constraint_penalty_weight × shortfall²` to the objective instead, where the shortfall is how far lifetime conversion falls below the floor (zero when it's above). In testing, near the baseline a uniform price change traded about 0.1 years of profit per percentage point of conversion. At a weight of 1e5, a 0.1-point shortfall costs as much as a full point of shortfall would earn, so the optimum ends up within a few thousandths of a point of the floor. Check `constraint_gap` in the summary: a small negative gap is the penalty at work.

**Settings and runtime.** Each simulation takes the time printed in section 5. `maxfev` caps the number of simulations, so it sets the runtime directly: 300 is about 25 minutes at the 10% sample. `xatol` (0.1% on a multiplier) and `fatol` (0.0001 years of profit) end the search earlier once the simplex has converged. If the cap is reached first, the summary reports `Maximum number of function evaluations has been exceeded` and the result is the best candidate so far. Raise `maxfev` for a tighter answer.

`bounds` keeps each table value within ±10%, as in notebook 02. A vehicle's multiplier is the product of two table values, so it can move by up to about 20%.

`enbp_ceiling=1.0` would add the UK GIPP rule, keeping each year's average renewal multiplier at or below the new-business equivalent. It isn't applied here.

In [ ]:
BOUNDS = (0.9, 1.1)
PENALTY_WEIGHT = 1e5
NELDER_MEAD_OPTIONS = {"maxfev": 300, "xatol": 1e-3, "fatol": 1e-4}

start = time.perf_counter()
result = optimizer.optimize(
    target_constraint=float(baseline_conversion),
    constraint_kind="floor",
    bounds=BOUNDS,
    method="Nelder-Mead",
    constraint_penalty_weight=PENALTY_WEIGHT,
    optimizer_options=NELDER_MEAD_OPTIONS,
    progress="bar",
)
print(f"optimised in {(time.perf_counter() - start) / 60:.1f} minutes")

result["summary"].T

In [ ]:
optimized = result["summary"].iloc[0]

print(f"Lifetime conversion: {baseline_conversion:.2%} -> {optimized['constraint']:.2%}")
print(f"Five-year NPV:       ${baseline_npv:,.0f} -> ${optimized['npv']:,.0f} ({optimized['npv'] / baseline_npv - 1:+.2%})")

`to_rating_tables` decodes the optimal multipliers back into the layer's table format. The vehicle counts are for the year-0 book, new business and renewals together.

In [ ]:
optimized_tables = optimizer.to_rating_tables()

year0 = pd.concat([scored_nb["risk"], scored_rn["risk"]], ignore_index=True)

for name, table in optimized_tables.items():
    key = table.columns[0]
    exposure = year0.groupby(key, observed=True).agg(
        vehicles=("risk.risk_number", "count"),
        renewal_share=("risk.transaction_type", lambda t: (t == "RN").mean()),
    )
    print(f"\n{name}")
    display(table.join(exposure, on=key).round(4))

### Year by year

The constraint is on lifetime conversion, so the optimiser is free to move conversion between years, and between new business and renewals, wherever that adds NPV.

In [ ]:
per_year = pd.concat(
    {
        "Baseline": baseline["per_period_summary"].set_index("period"),
        "Optimised": optimizer.output.per_period_summary.set_index("period"),
    },
    axis=1,
)
per_year.loc[:, (slice(None), ["conversion", "volume", "npv", "price_multiplier", "renewal_price_multiplier"])].round(4)

In [ ]:
BASELINE_COLOR = "#eb6834"
OPTIMISED_COLOR = "#2a78d6"
TEXT_COLOR = "#52514e"
GRID_COLOR = "#e4e3df"


def style_axis(ax, title):
    ax.set_title(title, loc="left")
    ax.grid(color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=TEXT_COLOR)


years = per_year.index.to_numpy()
fig, (ax_npv, ax_conv) = plt.subplots(1, 2, figsize=(12, 4.5))

width = 0.38
ax_npv.bar(years - width / 2, per_year[("Baseline", "npv")], width, color=BASELINE_COLOR, label="Baseline")
ax_npv.bar(years + width / 2, per_year[("Optimised", "npv")], width, color=OPTIMISED_COLOR, label="Optimised")
ax_npv.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"${y / 1e6:,.1f}m"))
ax_npv.set_xlabel("Year", color=TEXT_COLOR)
ax_npv.set_ylabel("Discounted expected profit", color=TEXT_COLOR)
style_axis(ax_npv, "NPV by year")
ax_npv.legend(frameon=False)

ax_conv.plot(years, per_year[("Baseline", "conversion")], color=BASELINE_COLOR, linewidth=2, marker="o", label="Baseline")
ax_conv.plot(years, per_year[("Optimised", "conversion")], color=OPTIMISED_COLOR, linewidth=2, marker="o", label="Optimised")
ax_conv.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax_conv.set_xlabel("Year", color=TEXT_COLOR)
ax_conv.set_ylabel("Conversion rate", color=TEXT_COLOR)
style_axis(ax_conv, "Conversion by year")
ax_conv.legend(frameon=False)

for ax in (ax_npv, ax_conv):
    ax.set_xticks(years)

plt.tight_layout()
plt.show()

### New business and renewals

The same tables price both cohorts, but the cohorts respond differently: a renewal's conversion depends on its price change and is limited by the cap and floor, while a new-business quote only competes on price. Splitting the simulated book by cohort shows where the optimiser found its NPV.

In [ ]:
def by_cohort(history):
    weighted = history.assign(
        weighted_conversion=history["sample_weights"] * history["risk.conversion_rate"],
        weighted_multiplier=history["sample_weights"] * history["risk.demand_multiplier"],
    )
    totals = weighted.groupby("__cohort").agg(
        weight=("sample_weights", "sum"),
        volume=("weighted_conversion", "sum"),
        multiplier=("weighted_multiplier", "sum"),
        npv=("expected_npv", "sum"),
    )
    return pd.DataFrame({
        "conversion": totals["volume"] / totals["weight"],
        "price_multiplier": totals["multiplier"] / totals["weight"],
        "volume": totals["volume"],
        "npv": totals["npv"],
    }).rename(index={"nb": "New business", "rn": "Renewals"})


pd.concat(
    {"Baseline": by_cohort(baseline["output_dataframe"]), "Optimised": by_cohort(result["output_dataframe"])},
    axis=1,
).round(4)

## 7. Efficient frontier

Re-running the optimisation across a range of lifetime conversion targets traces the most NPV available at each level of conversion. The targets run between the conversion of a uniform 5% increase and a uniform 5% decrease. `constraint_kind="target"` makes each one an equality, so a converged point sits at its target. `constraint_gap` shows how close a capped one got.

Each point is a full optimisation, so this is the slowest cell in the notebook. It uses only a few targets, each capped at a smaller `maxfev`: 3 × 120 simulations is about 30 minutes at the 10% sample. The equality target is penalised in the same way as the floor, but on both sides.

`calculate_efficient_frontier` would do the sweep in one call, but it drops any target whose solver doesn't report convergence. With `maxfev` capped, that can be every target. The loop below calls `optimize` for each target instead, keeps every result, and flags the ones that stopped at the cap. `save_to_log=False` leaves the section 6 result in `optimizer.output`, so sections 8 and 9 still use it.

In [ ]:
FRONTIER_TARGETS = np.linspace(uniform_response.loc[1.05, "conversion"], uniform_response.loc[0.95, "conversion"], 3)
FRONTIER_OPTIONS = {**NELDER_MEAD_OPTIONS, "maxfev": 120}

start = time.perf_counter()
frontier_rows = []
for target in FRONTIER_TARGETS:
    point = optimizer.optimize(
        target_constraint=float(target),
        constraint_kind="target",
        bounds=BOUNDS,
        method="Nelder-Mead",
        constraint_penalty_weight=PENALTY_WEIGHT,
        optimizer_options=FRONTIER_OPTIONS,
        save_to_log=False,
        progress=False,
    )
    frontier_rows.append(point["summary"])
    print(f"target {target:.2%}: NPV ${point['summary']['npv'].iloc[0]:,.0f}")
print(f"frontier in {(time.perf_counter() - start) / 60:.1f} minutes")

frontier = pd.concat(frontier_rows, ignore_index=True)
frontier[["target_constraint", "constraint", "constraint_gap", "npv", "price_multiplier", "result_nit", "result_message"]].round(4)

Changing every price by the same factor is one of the solutions open to the optimiser, so the frontier should sit on or above the uniform line. Wherever it falls below, the solver stopped short of the best solution for that target.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    uniform_response["conversion"], uniform_response["npv"],
    color=BASELINE_COLOR, linewidth=2, marker="o", markersize=5, label="Uniform price change",
)
ax.plot(
    frontier["constraint"], frontier["npv"],
    color=OPTIMISED_COLOR, linewidth=2, marker="o", markersize=5, label="Optimised rating factors (frontier)",
)
ax.scatter([baseline_conversion], [baseline_npv], s=90, color="white", edgecolor=TEXT_COLOR, linewidth=2, zorder=3)
ax.annotate("Baseline", (baseline_conversion, baseline_npv), xytext=(-14, -18), textcoords="offset points", ha="right", color=TEXT_COLOR)
ax.scatter([optimized["constraint"]], [optimized["npv"]], s=90, color=OPTIMISED_COLOR, edgecolor="white", linewidth=2, zorder=4)
ax.annotate("Optimised", (optimized["constraint"], optimized["npv"]), xytext=(12, 8), textcoords="offset points", color=TEXT_COLOR)

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"${y / 1e6:,.2f}m"))
ax.set_xlabel("Lifetime conversion rate", color=TEXT_COLOR)
ax.set_ylabel("Five-year NPV of expected profit", color=TEXT_COLOR)
style_axis(ax, "Five-year NPV vs lifetime conversion")
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

## 8. Risk: the distribution of NPV

The simulation carries each vehicle's survival as a fractional weight, which gives the *expected* NPV. `monte_carlo_npv` instead samples whether each vehicle renews in each year, and once a vehicle lapses it earns nothing afterwards. Repeating that `n_sims` times gives the spread of NPV around its expected value, and its tail: `var` is the 5th-percentile NPV and `cvar` the average NPV in the worst 5% of runs.

It scores the pipeline once for a given set of multipliers and samples on top of that, so it's fast. It's a reporting tool, not part of the optimisation. The sample mean should match the deterministic NPV to within sampling error.

Running it for both the baseline and the optimised tables shows whether the extra NPV comes with extra risk. On a sampled book the spread is wider, relative to the NPV, than it would be on the full book.

In [ ]:
mc_baseline = optimizer.monte_carlo_npv(optimizer.input.initial_multipliers, n_sims=2000, seed=0)
mc_optimized = optimizer.monte_carlo_npv(n_sims=2000, seed=0)  # defaults to the last optimize() result

pd.concat({"Baseline": mc_baseline.summary.T[0], "Optimised": mc_optimized.summary.T[0]}, axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

bins = np.histogram_bin_edges(np.concatenate([mc_baseline.samples, mc_optimized.samples]), bins=50)
for mc, color, label in [(mc_baseline, BASELINE_COLOR, "Baseline"), (mc_optimized, OPTIMISED_COLOR, "Optimised")]:
    ax.hist(mc.samples, bins=bins, color=color, alpha=0.55, label=label)
    ax.axvline(mc.deterministic_npv, color=color, linewidth=2)
    ax.axvline(mc.var, color=color, linewidth=1.5, linestyle="--")

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x / 1e6:,.2f}m"))
ax.set_xlabel("Five-year NPV (solid: expected; dashed: 5% VaR)", color=TEXT_COLOR)
ax.set_ylabel("Simulations", color=TEXT_COLOR)
style_axis(ax, "Monte-Carlo NPV distribution")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

## 9. Deploy the optimised rating factors

`to_layer` rebuilds the `RatingTablesLayer` from the optimised tables, keeping the original layer's name, formula, defaults, dependencies and output column. Swapping it into a copy of the pipeline leaves the baseline untouched for comparison.

In [ ]:
optimized_engine = deepcopy(pricing_engine)
optimized_engine.remove_parameter(LAYER_NAME)
optimized_engine.add_parameter(optimizer.to_layer())

for name, table in optimized_engine.layers[LAYER_NAME].rating_tables.items():
    print(f"\n{name}")
    display(table.round(4))

### Re-score year 0

Year 0 is next year's quotes, the book the deployed tables price first. Scoring it with the optimised pipeline should reproduce the optimiser's year-0 figures exactly. There's no interpolation, as the optimiser scored this same pipeline. Later years can only be checked by simulation, since they depend on which customers renew.

In [ ]:
optimized_nb = optimized_engine.predict(data_nb)
optimized_rn = optimized_engine.predict(data_rn)

rescored_year0 = pd.concat([optimized_nb["risk"], optimized_rn["risk"]], ignore_index=True)
simulated_year0 = optimizer.output.per_period_summary.set_index("period").loc[0]
discount_year0 = (1 + DISCOUNT_RATE) ** 0.5

pd.DataFrame({
    "Optimiser (year 0)": {
        "Conversion rate": simulated_year0["conversion"],
        "Discounted expected profit": simulated_year0["npv"],
    },
    "Re-scored": {
        "Conversion rate": rescored_year0["risk.conversion_rate"].mean(),
        "Discounted expected profit": rescored_year0["risk.reporting_expected_profit"].sum() / discount_year0,
    },
}).round(4)

In [ ]:
pd.DataFrame({
    "New business: baseline": book_kpis(scored_nb),
    "New business: optimised": book_kpis(optimized_nb),
    "Renewals: baseline": book_kpis(scored_rn),
    "Renewals: optimised": book_kpis(optimized_rn),
}).round(4)

## 10. Save the optimised pipeline

Saved with `cloudpickle` beside the baseline, in the same way as notebooks 01–03, under its own name so it doesn't replace notebook 02's result. Load it with `cloudpickle.load` and call `load_context` before scoring.

In [ ]:
OPTIMIZED_PATH = MODELS_DIR / "rating_engines" / "pricing_pipeline_rating_factor_multi_period_optimized.pkl"

with OPTIMIZED_PATH.open("wb") as file:
    cloudpickle.dump(optimized_engine, file)

print(f"Pipeline saved to {OPTIMIZED_PATH}")